In [1]:
# @title 1. Dependencias
# Sit.2 — SAE post-entrenamiento (features del checkpoint `sit2_geovision_clip`)

%pip install -q huggingface_hub open_clip_torch zarr numcodecs pandas pyarrow numpy scipy scikit-learn
%pip install -q torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 319.6/319.6 kB 24.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 45.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.1 MB/s eta 0:00:00


In [2]:
# @title 2. Configurar rutas + traer artefactos de sit2_geovision_clip
# Ruta manual (Kaggle/local) si no hay Colab/Drive:
CKPT_PATH_MANUAL = "/kaggle/input/pesos-pt/best.pt"

HF_REPO_ID = "Slucu-0310/geovision-cali-sit2"
SEED = 42

import os, sys, json, hashlib, shutil, zipfile
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm

np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

OUT_DIR = Path("/content/runs/sit2_sae")
CLIP_DIR = OUT_DIR / "clip_modelo"
EMB_DIR = OUT_DIR / "embeddings"
ANALISIS_DIR = OUT_DIR / "analisis"
for d in (OUT_DIR, CLIP_DIR, EMB_DIR, ANALISIS_DIR):
    d.mkdir(parents=True, exist_ok=True)

CLIP_FILES = (
    "best.pt",
    "training_logs.json",
    "training_history.json",
    "training_curves.png",
    "metadata.json",
    "hashes.sha256",
)

def _copy_if_newer(src: Path, dst: Path) -> bool:
    if not src.is_file():
        return False
    if dst.is_file() and dst.stat().st_mtime >= src.stat().st_mtime:
        return False
    shutil.copy2(src, dst)
    return True

def sync_clip_artifacts() -> list[str]:
    """Copia salidas de sit2_geovision_clip.ipynb a clip_modelo/."""
    copied = []
    sources = [
        Path("/content/runs/sit2_recall_final"),
        Path("/content/geovision_clip_modelo_final"),
        Path("/content/drive/MyDrive/geovision-cali/sit2_geovision_clip"),
    ]
    zip_clip = Path("/content/geovision_clip_modelo_final.zip")
    if zip_clip.is_file() and not (CLIP_DIR / "best.pt").is_file():
        with zipfile.ZipFile(zip_clip) as zf:
            zf.extractall(CLIP_DIR)
            copied.append(str(zip_clip))

    try:
        from google.colab import drive
        if not Path("/content/drive").is_dir():
            drive.mount("/content/drive", force_remount=False)
    except Exception:
        pass

    for root in sources:
        if not root.exists():
            continue
        for name in CLIP_FILES:
            if _copy_if_newer(root / name, CLIP_DIR / name):
                copied.append(name)
        curvas_src = root / "curvas"
        if curvas_src.is_dir():
            shutil.copytree(curvas_src, CLIP_DIR / "curvas", dirs_exist_ok=True)
            copied.append("curvas/")

    manual = Path(CKPT_PATH_MANUAL)
    if manual.is_file() and _copy_if_newer(manual, CLIP_DIR / "best.pt"):
        copied.append("best.pt (manual)")

    if not (CLIP_DIR / "best.pt").is_file() and Path("/kaggle/input").exists():
        cand = sorted(Path("/kaggle/input").rglob("best.pt"), key=os.path.getmtime, reverse=True)
        if cand and _copy_if_newer(cand[0], CLIP_DIR / "best.pt"):
            copied.append(f"best.pt ({cand[0]})")

    if not (CLIP_DIR / "best.pt").is_file() and Path("/content/runs").exists():
        for p in sorted(Path("/content/runs").rglob("best.pt"), key=os.path.getmtime, reverse=True):
            if "sit2_sae" in str(p):
                continue
            if _copy_if_newer(p, CLIP_DIR / "best.pt"):
                copied.append(f"best.pt ({p})")
                break

    return copied

copied = sync_clip_artifacts()
CKPT = CLIP_DIR / "best.pt"
if not CKPT.is_file():
    raise FileNotFoundError(
        "No hay best.pt del notebook sit2_geovision_clip. "
        "Ejecuta entrenar + export en Colab, monta Drive, o sube el zip/dataset a Kaggle."
    )
print(f"Checkpoint CLIP: {CKPT} ({CKPT.stat().st_size / 1e6:.1f} MB)")
print(f"clip_modelo/: {[p.name for p in sorted(CLIP_DIR.iterdir())]}")
if copied:
    print("Sincronizado:", sorted(set(copied)))

Device: cuda
Mounted at /content/drive
Checkpoint CLIP: /content/runs/sit2_sae/clip_modelo/best.pt (635.3 MB)
clip_modelo/: ['best.pt', 'hashes.sha256', 'metadata.json', 'training_logs.json']
Sincronizado: ['best.pt', 'hashes.sha256', 'metadata.json', 'training_logs.json']


In [3]:
# @title 3. Descargar dataset desde HF
from huggingface_hub import snapshot_download
import pandas as pd
import zarr

DATA_DIR = Path("/content/dataset_sit2")
if not (DATA_DIR / "metadatos.parquet").is_file():
    print("Descargando dataset desde HF...")
    try:
        from google.colab import userdata
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    except:
        pass
    from huggingface_hub import login
    if os.environ.get("HF_TOKEN"):
        login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)
    snapshot_download(
        repo_id=HF_REPO_ID, repo_type="dataset",
        local_dir=str(DATA_DIR), local_dir_use_symlinks=False
    )
else:
    print("Dataset ya descargado")

# Cargar metadatos
df = pd.read_parquet(DATA_DIR / "metadatos.parquet")
print(f"Pares: {len(df)}")

# Cargar tiles
store = zarr.storage.LocalStore(str(DATA_DIR / "tiles.zarr"))
tiles_z = zarr.open(store, mode="r")
if isinstance(tiles_z, zarr.Group):
    tiles_z = tiles_z["tiles"]
print(f"Tiles shape: {tiles_z.shape}")

# Split estratificado (misma semilla que notebook principal)
from sklearn.model_selection import train_test_split
if "split" not in df.columns:
    ti, tmp = train_test_split(np.arange(len(df)), test_size=0.30, stratify=df["clase"].values, random_state=42)
    vi, te = train_test_split(tmp, test_size=0.50, stratify=df.iloc[tmp]["clase"].values, random_state=42)
    df["split"] = "train"; df.loc[vi, "split"] = "val"; df.loc[te, "split"] = "test"

CLASES = ["contaminacion_alta_NO2", "contaminacion_alta_SO2", "ozono_anomalo",
          "vegetacion_densa", "suelo_urbano"]
CLASE_A_IDX = {c: i for i, c in enumerate(CLASES)}

# Normalizacion stats
rng = np.random.default_rng(42)
ns = min(512, int(tiles_z.shape[0]))
ix = rng.choice(int(tiles_z.shape[0]), ns, replace=False)
sm = np.stack([np.asarray(tiles_z[i], dtype=np.float32) for i in ix])
bm = sm.mean(axis=(0,2,3)).astype(np.float32)
bs = np.maximum(sm.std(axis=(0,2,3)), 1e-3).astype(np.float32)
print(f"Stats computed: mean={bm.shape}, std={bs.shape}")

Descargando dataset desde HF...


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `snapshot_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


Fetching 43 files:   0%|          | 0/43 [00:00<?, ?it/s]

Pares: 2263
Tiles shape: (2263, 13, 64, 64)
Stats computed: mean=(13,), std=(13,)


In [4]:
# @title 4. Cargar checkpoint + extraer features
import open_clip
from huggingface_hub import hf_hub_download

# --- Cargar RemoteCLIP ---
cm, _, _ = open_clip.create_model_and_transforms("ViT-B-32", pretrained=None)
cp_remote = hf_hub_download("chendelong/RemoteCLIP", "RemoteCLIP-ViT-B-32.pt",
                            cache_dir="/content/checkpoints")
cm.load_state_dict(torch.load(cp_remote, map_location="cpu"), strict=False)

# --- Adaptar conv1 a 12 canales ---
oc = cm.visual.conv1
nc = nn.Conv2d(12, oc.out_channels, oc.kernel_size,
               stride=oc.stride, padding=oc.padding, bias=False)
with torch.no_grad():
    ir, ig, ib = 3, 2, 1
    w = nc.weight.data
    w[:,ir] = oc.weight[:,0]
    w[:,ig] = oc.weight[:,1]
    w[:,ib] = oc.weight[:,2]
    wm = oc.weight.mean(dim=1, keepdim=False)
    for b in range(12):
        if b not in (ir, ig, ib):
            w[:,b] = wm * (3.0/12)
    nc.weight.copy_(w)
cm.visual.conv1 = nc

# --- Cargar pesos fine-tuned ---
state = torch.load(CKPT, map_location="cpu")
if "cm" in state:
    cm.load_state_dict(state["cm"], strict=False)
elif "clip_model" in state:
    cm.load_state_dict(state["clip_model"], strict=False)
else:
    # checkpoint solo con visual encoder?
    cm.load_state_dict(state, strict=False)

cm = cm.to(DEVICE)
cm.eval()
print("Checkpoint cargado. Visual encoder listo.")

# --- Extraer features de todos los tiles ---
BATCH_SIZE = 64
print("Extrayendo features de todos los tiles...")
all_feats = []
all_labels = []

with torch.no_grad():
    for i in tqdm(range(0, len(df), BATCH_SIZE), desc="Features"):
        batch_idx = slice(i, min(i + BATCH_SIZE, len(df)))
        batch_tiles = []
        for j in range(batch_idx.start, batch_idx.stop):
            t = np.asarray(tiles_z[j], dtype=np.float32)
            t = (t - bm.reshape(13,1,1)) / bs.reshape(13,1,1)
            batch_tiles.append(t[:12])  # 12 bandas
        tiles = torch.from_numpy(np.stack(batch_tiles)).float().to(DEVICE)
        tiles_224 = F.interpolate(tiles, size=(224, 224), mode="bilinear", align_corners=False)
        feats = F.normalize(cm.visual(tiles_224), dim=-1)
        all_feats.append(feats.cpu())
        all_labels.extend(df.iloc[batch_idx]["clase"].values)

features = torch.cat(all_feats, dim=0)  # (N, 512)
labels = np.array(all_labels)
print(f"Features extraidas: {features.shape}")

feat_path = EMB_DIR / "visual_512.pt"
torch.save({
    "features": features,
    "labels": labels,
    "clases": CLASES,
    "df_index": df.index.tolist(),
    "split": df["split"].tolist() if "split" in df.columns else None,
    "bm": bm,
    "bs": bs,
    "checkpoint_clip": str(CKPT),
}, feat_path)
legacy = OUT_DIR / "features.pt"
torch.save({"features": features, "labels": labels, "bm": bm, "bs": bs}, legacy)

manifest = {
    "n_samples": int(len(df)),
    "feature_dim": 512,
    "dataset": HF_REPO_ID,
    "files": {
        "visual_embeddings": str(feat_path),
        "legacy_features": str(legacy),
        "clip_checkpoint": str(CLIP_DIR / "best.pt"),
    },
}
(EMB_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print(f"Embeddings: {feat_path}")
print(f"Manifest: {EMB_DIR / 'manifest.json'}")

print(f"Feature mean abs: {features.abs().mean():.4f}")
print(f"Feature std: {features.std():.4f}")

RemoteCLIP-ViT-B-32.pt:   0%|          | 0.00/605M [00:00<?, ?B/s]

Checkpoint cargado. Visual encoder listo.
Extrayendo features de todos los tiles...


Features:   0%|          | 0/36 [00:00<?, ?it/s]

Features extraidas: torch.Size([2263, 512])
Embeddings: /content/runs/sit2_sae/embeddings/visual_512.pt
Manifest: /content/runs/sit2_sae/embeddings/manifest.json
Feature mean abs: 0.0282
Feature std: 0.0442


In [5]:
# @title 5. Entrenar Sparse Autoencoder

class SAE(nn.Module):
    """Sparse Autoencoder: 512 -> ReLU -> 512."""
    def __init__(self, d_model=512):
        super().__init__()
        self.encoder = nn.Linear(d_model, d_model, bias=False)
        self.decoder = nn.Linear(d_model, d_model, bias=False)
        nn.init.xavier_uniform_(self.encoder.weight)
        nn.init.xavier_uniform_(self.decoder.weight)

    def forward(self, x):
        h = F.relu(self.encoder(x))
        x_hat = self.decoder(h)
        return x_hat, h

    def normalize_decoder(self):
        """Normaliza filas del decoder a norma unitaria."""
        with torch.no_grad():
            self.decoder.weight.data = F.normalize(self.decoder.weight.data, dim=-1)

# Hiperparametros
LR = 1e-3
LAMBDA_L1 = 5e-3  # aumentado para mejor sparsity  # peso de regularizacion L1
EPOCHS = 4000
BATCH_SIZE = 512

sae = SAE(512).to(DEVICE)
opt = torch.optim.AdamW(sae.parameters(), lr=LR, weight_decay=1e-6)
print(f"SAE params: {sum(p.numel() for p in sae.parameters())}")

# Dataset en GPU (features ya extraidas)
X = features.to(DEVICE)
dataset = torch.utils.data.TensorDataset(X)
loader = torch.utils.data.DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

# Entrenamiento
best_mse = float("inf")
history = []

print(f"Entrenando SAE por {EPOCHS} epochs...")
for ep in range(EPOCHS):
    sae.train()
    total_mse = 0.0
    total_l1 = 0.0
    total_sparsity = 0.0
    n_batches = 0

    for (x_batch,) in loader:
        x_hat, h = sae(x_batch)

        # Reconstruction loss
        mse = F.mse_loss(x_hat, x_batch)

        # L1 sparsity penalty
        l1 = h.abs().mean()

        loss = mse + LAMBDA_L1 * l1

        opt.zero_grad()
        loss.backward()
        opt.step()

        sae.normalize_decoder()

        # Sparsity: fraccion de activaciones < 0.01
        sparsity = (h.abs() < 0.01).float().mean()

        total_mse += mse.item()
        total_l1 += l1.item()
        total_sparsity += sparsity.item()
        n_batches += 1

    avg_mse = total_mse / n_batches
    avg_l1 = total_l1 / n_batches
    avg_sparsity = total_sparsity / n_batches

    history.append({"epoch": ep, "mse": avg_mse, "l1": avg_l1, "sparsity": avg_sparsity})

    if avg_mse < best_mse:
        best_mse = avg_mse
        torch.save({"sae": sae.state_dict(), "mse": avg_mse, "sparsity": avg_sparsity},
                   OUT_DIR / "sae_best.pt")

    # Ajuste adaptativo de L1 para alcanzar sparsity target
    SPARSITY_TARGET = 0.75  # queremos al menos 75% de neuronas inactivas
    if ep > 200 and ep % 100 == 0:
        if avg_sparsity < SPARSITY_TARGET - 0.05 and LAMBDA_L1 < 0.1:
            LAMBDA_L1 *= 1.2
            for pg in opt.param_groups:
                pass  # no necesitamos ajustar lr
        elif avg_sparsity > SPARSITY_TARGET + 0.1 and LAMBDA_L1 > 1e-4:
            LAMBDA_L1 *= 0.8

    if ep % 400 == 0 or ep == EPOCHS - 1:
        print(f"Ep {ep:4d} | MSE={avg_mse:.6f} | L1={avg_l1:.4f} | Sparsity={avg_sparsity:.4f} | lambda={LAMBDA_L1:.2e}")

# Guardar historial de entrenamiento para curvas
with open(OUT_DIR / "training_history.json", "w") as f:
    json.dump(history, f, indent=2)
print(f"Historial guardado: {OUT_DIR / 'training_history.json'}")

print(f"\nEntrenamiento completo.")
print(f"Best MSE: {best_mse:.6f}")

SAE params: 524288
Entrenando SAE por 4000 epochs...
Ep    0 | MSE=0.001841 | L1=0.0152 | Sparsity=0.6266 | lambda=5.00e-03
Ep  400 | MSE=0.000028 | L1=0.0086 | Sparsity=0.8053 | lambda=5.00e-03
Ep  800 | MSE=0.000020 | L1=0.0112 | Sparsity=0.7925 | lambda=5.00e-03
Ep 1200 | MSE=0.000019 | L1=0.0100 | Sparsity=0.7978 | lambda=5.00e-03
Ep 1600 | MSE=0.000017 | L1=0.0119 | Sparsity=0.7844 | lambda=5.00e-03
Ep 2000 | MSE=0.000017 | L1=0.0103 | Sparsity=0.7981 | lambda=5.00e-03
Ep 2400 | MSE=0.000017 | L1=0.0094 | Sparsity=0.8052 | lambda=5.00e-03
Ep 2800 | MSE=0.000017 | L1=0.0094 | Sparsity=0.8061 | lambda=5.00e-03
Ep 3200 | MSE=0.000017 | L1=0.0103 | Sparsity=0.8042 | lambda=5.00e-03
Ep 3600 | MSE=0.000017 | L1=0.0112 | Sparsity=0.8009 | lambda=5.00e-03
Ep 3999 | MSE=0.000018 | L1=0.0082 | Sparsity=0.8134 | lambda=5.00e-03
Historial guardado: /content/runs/sit2_sae/training_history.json

Entrenamiento completo.
Best MSE: 0.000015


In [6]:
# @title 6. Analisis de neuronas por clase + correlacion + top activaciones
import json, math
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm

CLASES = ["contaminacion_alta_NO2", "contaminacion_alta_SO2", "ozono_anomalo",
          "vegetacion_densa", "suelo_urbano"]

# Cargar mejor SAE
ckpt_sae = torch.load(OUT_DIR / "sae_best.pt", map_location="cpu")
sae.load_state_dict(ckpt_sae["sae"])
sae.eval()
print(f"Cargado SAE best: MSE={ckpt_sae['mse']:.6f}, Sparsity={ckpt_sae['sparsity']:.4f}")

# Activar modo test para evaluacion sin dropout etc
sae.eval()
X_dev = features.to(DEVICE)
with torch.no_grad():
    _, h_all = sae(X_dev)

h_all = h_all.cpu().numpy()  # (N, 512)
print(f"Activaciones: {h_all.shape}")

# === 1. Activacion promedio por clase ===
class_activation = {}
for cls in CLASES:
    mask = labels == cls
    if mask.sum() == 0:
        continue
    acts = np.abs(h_all[mask])  # (N_clase, 512)
    mean_act = acts.mean(axis=0)  # (512,)
    class_activation[cls] = mean_act

print("\n=== Top 5 neuronas por clase ===")
top_neurons_per_class = {}
for cls in CLASES:
    mean_act = class_activation[cls]
    top_idx = np.argsort(mean_act)[::-1][:10]
    top_vals = mean_act[top_idx]
    top_neurons_per_class[cls] = list(zip(top_idx.tolist(), top_vals.tolist()))
    print(f"\n{cls}:")
    for nid, val in zip(top_idx[:5], top_vals[:5]):
        print(f"  Neurona {nid:3d} | activacion media = {val:.4f}")

# === 2. Correlacion con variables numericas ===
print("\n=== Correlacion neuronas vs variables ambientales ===")
num_cols = [c for c in df.columns if c in ('ndvi','bsi','no2','so2','o3',
    'valid_ratio','frac_nubes_scl','frac_claros_scl','frac_nodata_scl')]
if num_cols:
    corr_results = []
    for col in num_cols:
        vals = df[col].values.astype(np.float32)
        # Correlacion de cada neurona con esta variable
        corrs = np.array([np.corrcoef(h_all[:,j], vals)[0,1] for j in range(h_all.shape[1])])
        top_corr_idx = np.argsort(np.abs(corrs))[::-1][:5]
        corr_results.append({"variable": col, "top_neurons": [
            {"neurona": int(j), "corr": float(corrs[j])} for j in top_corr_idx
        ]})
        print(f"\n{col}:")
        for j in top_corr_idx[:3]:
            print(f"  Neurona {j:3d} | r = {corrs[j]:+.4f}")

    with open(ANALISIS_DIR / "correlaciones_neuronas.json", "w") as f:
        json.dump(corr_results, f, indent=2, ensure_ascii=False)

# === 3. Top 3 imagenes que mas activan cada neurona top ===
print("\n=== Top imagenes por neurona ===")
top_images = {}
h_abs = np.abs(h_all)
for cls in CLASES:
    top_neurons = [nid for nid, val in top_neurons_per_class[cls][:5]]
    cls_mask = labels == cls
    cls_indices = np.where(cls_mask)[0]
    for nid in top_neurons:
        # Entre las muestras de esta clase, cual activa mas esta neurona
        cls_acts = h_abs[cls_mask, nid]
        top3_local = np.argsort(cls_acts)[::-1][:3]
        top3_global = cls_indices[top3_local].tolist()
        top_images[f"neurona_{nid}"] = {
            "clase_mas_asociada": cls,
            "top3_indices": top3_global,
            "activaciones": cls_acts[top3_local].tolist()
        }

with open(ANALISIS_DIR / "top_activations_per_neuron.json", "w") as f:
    json.dump(top_images, f, indent=2, ensure_ascii=False)

latent_path = EMB_DIR / "sae_latent_512.pt"
torch.save({"latent": torch.from_numpy(h_all), "labels": labels}, latent_path)
print(f"Latentes SAE: {latent_path}")
if (EMB_DIR / "manifest.json").is_file():
    m = json.loads((EMB_DIR / "manifest.json").read_text(encoding="utf-8"))
    m["files"]["sae_latent"] = str(latent_path)
    (EMB_DIR / "manifest.json").write_text(json.dumps(m, indent=2), encoding="utf-8")

# === 4. Heatmap de activacion clase vs neurona ===
print("\n=== Generando heatmap ===")
n_top = 20
# Top 20 neuronas mas discriminativas entre clases
all_means = np.stack([class_activation[c] for c in CLASES], axis=0)  # (5, 512)
# Varianza entre clases -> neuronas mas discriminativas
between_var = all_means.var(axis=0)  # (512,)
top_disc_idx = np.argsort(between_var)[::-1][:n_top]

fig, ax = plt.subplots(figsize=(10, 6))
heatmap_data = all_means[:, top_disc_idx]
im = ax.imshow(heatmap_data, aspect='auto', cmap='viridis')
ax.set_yticks(range(len(CLASES)))
ax.set_yticklabels(CLASES, fontsize=9)
ax.set_xticks(range(n_top))
ax.set_xticklabels([str(j) for j in top_disc_idx], fontsize=8, rotation=45)
ax.set_xlabel("Neurona")
ax.set_title("Activacion promedio (top 20 neuronas mas discriminativas)")
plt.colorbar(im, label="Activacion media")
fig.tight_layout()
fig.savefig(str(ANALISIS_DIR / "heatmap_neurona_clase.png"), dpi=150)
print(f"Heatmap guardado: {ANALISIS_DIR / 'heatmap_neurona_clase.png'}")

# === 5. Histograma de distribucion de sparsity ===
print("\n=== Sparsity por sample ===")
sparsity_per_sample = (h_abs < 0.01).mean(axis=1)
fig2, ax2 = plt.subplots(figsize=(8, 4))
ax2.hist(sparsity_per_sample, bins=50, alpha=0.7)
ax2.axvline(sparsity_per_sample.mean(), color='r', linestyle='--',
            label=f"Media = {sparsity_per_sample.mean():.3f}")
ax2.set_xlabel("Sparsity (fraccion inactiva)")
ax2.set_ylabel("Frecuencia")
ax2.set_title("Distribucion de sparsity por muestra")
ax2.legend()
fig2.tight_layout()
fig2.savefig(str(ANALISIS_DIR / "sparsity_histogram.png"), dpi=150)
print(f"Histograma guardado")

resultados = {
    "top_neurons_per_class": {k: v[:5] for k, v in top_neurons_per_class.items()},
    "n_samples": int(len(labels)),
}
with open(ANALISIS_DIR / "resultados_sae.json", "w", encoding="utf-8") as f:
    json.dump(resultados, f, indent=2, ensure_ascii=False)

print(f"\nAnalisis en {ANALISIS_DIR}")
for sub in (ANALISIS_DIR, EMB_DIR, CLIP_DIR):
    print(f"  {sub.name}/:", [p.name for p in sorted(sub.iterdir())[:12]])

Cargado SAE best: MSE=0.000015, Sparsity=0.7579
Activaciones: (2263, 512)

=== Top 5 neuronas por clase ===

contaminacion_alta_NO2:
  Neurona  36 | activacion media = 2.4243
  Neurona 452 | activacion media = 1.9957
  Neurona 410 | activacion media = 0.2583
  Neurona  47 | activacion media = 0.0424
  Neurona  19 | activacion media = 0.0411

contaminacion_alta_SO2:
  Neurona 452 | activacion media = 2.2803
  Neurona  36 | activacion media = 2.1702
  Neurona 410 | activacion media = 0.2554
  Neurona 406 | activacion media = 0.0414
  Neurona  13 | activacion media = 0.0412

ozono_anomalo:
  Neurona 452 | activacion media = 2.3013
  Neurona  36 | activacion media = 2.2183
  Neurona 410 | activacion media = 0.2584
  Neurona  23 | activacion media = 0.0304
  Neurona 349 | activacion media = 0.0301

vegetacion_densa:
  Neurona 452 | activacion media = 2.6536
  Neurona  36 | activacion media = 1.8469
  Neurona 410 | activacion media = 0.2444
  Neurona 406 | activacion media = 0.0453
  Neurona

/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2922: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2923: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]



frac_claros_scl:
  Neurona 369 | r = +nan
  Neurona 368 | r = +nan
  Neurona 137 | r = +nan

frac_nodata_scl:
  Neurona  90 | r = +nan
  Neurona 357 | r = +nan
  Neurona 356 | r = +nan

ndvi:
  Neurona  56 | r = +nan
  Neurona 322 | r = +nan
  Neurona 321 | r = +nan

bsi:
  Neurona 321 | r = +nan
  Neurona 320 | r = +nan
  Neurona 319 | r = +nan

no2:
  Neurona 161 | r = +nan
  Neurona 162 | r = +nan
  Neurona 163 | r = +nan

so2:
  Neurona 161 | r = +nan
  Neurona 162 | r = +nan
  Neurona 163 | r = +nan

o3:
  Neurona 161 | r = +nan
  Neurona 162 | r = +nan
  Neurona 163 | r = +nan

=== Top imagenes por neurona ===
Latentes SAE: /content/runs/sit2_sae/embeddings/sae_latent_512.pt

=== Generando heatmap ===
Heatmap guardado: /content/runs/sit2_sae/analisis/heatmap_neurona_clase.png

=== Sparsity por sample ===
Histograma guardado

Analisis en /content/runs/sit2_sae/analisis
  analisis/: ['correlaciones_neuronas.json', 'heatmap_neurona_clase.png', 'resultados_sae.json', 'sparsity_histo

In [7]:
# @title 6. AFE + AFC: Scree plot, cargas rotadas, indices de ajuste
%pip install -q factor_analyzer

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path
import json
import torch
import torch.nn.functional as F
from sklearn.decomposition import PCA
from factor_analyzer import (
    FactorAnalyzer,
    ConfirmatoryFactorAnalyzer,
    ModelSpecificationParser,
    calculate_kmo,
    calculate_bartlett_sphericity,
)
from numpy.linalg import slogdet, inv

CLASES = ["contaminacion_alta_NO2", "contaminacion_alta_SO2", "ozono_anomalo",
          "vegetacion_densa", "suelo_urbano"]

# Cargar SAE y activaciones
ckpt_sae = torch.load(OUT_DIR / "sae_best.pt", map_location="cpu")
sae.load_state_dict(ckpt_sae["sae"])
sae.eval()
X_dev = features.to(DEVICE)
with torch.no_grad():
    _, h_all = sae(X_dev)
h_all = h_all.cpu().numpy()  # (N, 512)
print(f"Activaciones: {h_all.shape}")

# Seleccionar las N neuronas mas discriminativas (mayor varianza entre clases)
N_FA = 25  # neuronas para el analisis factorial
class_means = []
for cls in CLASES:
    mask = labels == cls
    class_means.append(np.abs(h_all[mask]).mean(axis=0))
class_means = np.stack(class_means)  # (5, 512)
between_var = class_means.var(axis=0)
top_idx = np.argsort(between_var)[::-1][:N_FA]
print(f"Top {N_FA} neuronas discriminativas: {top_idx.tolist()}")

X_fa = h_all[:, top_idx].astype(np.float64)
std_fa = X_fa.std(axis=0)
std_fa[std_fa < 1e-8] = 1.0
X_fa = (X_fa - X_fa.mean(axis=0)) / std_fa
X_fa = np.nan_to_num(X_fa, nan=0.0, posinf=0.0, neginf=0.0)

# === 1. SCREE PLOT (PCA) ===
pca = PCA()
pca.fit(X_fa)
ev = pca.explained_variance_
evr = pca.explained_variance_ratio_

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(range(1, len(ev)+1), ev, 'bo-', markersize=4)
ax1.axhline(y=1.0, color='r', linestyle='--', alpha=0.5, label='Kaiser criterion (eigenvalue=1)')
ax1.set_xlabel("Factor"); ax1.set_ylabel("Eigenvalue")
ax1.set_title("Scree plot"); ax1.legend(); ax1.grid(True, alpha=0.3)

n_factors_kaiser = int((ev >= 1.0).sum())
ax1.axvline(x=n_factors_kaiser + 0.5, color='g', linestyle=':', alpha=0.7,
            label=f'Kaiser: {n_factors_kaiser} factores')
ax1.legend()

ax2.bar(range(1, len(evr)+1), evr, alpha=0.7)
ax2.plot(range(1, len(evr)+1), np.cumsum(evr), 'ro-', markersize=4)
ax2.set_xlabel("Factor"); ax2.set_ylabel("Varianza explicada")
ax2.set_title("Varianza explicada (acumulada)")
ax2.grid(True, alpha=0.3)
for i in range(min(5, len(evr))):
    ax2.text(i+1, evr[i]+0.01, f'{evr[i]:.1%}', ha='center', fontsize=8)

fig.tight_layout()
fig.savefig(str(ANALISIS_DIR / "scree_plot.png"), dpi=150)
print(f"Scree plot: {ANALISIS_DIR / 'scree_plot.png'}")

# === 2. PRUEBAS DE IDONEIDAD ===
print("\n=== Pruebas de idoneidad ===")
kmo_all, kmo_total = calculate_kmo(X_fa)
print(f"KMO global: {kmo_total:.4f}")

chi2_stat, p_val = calculate_bartlett_sphericity(X_fa)
print(f"Bartlett sphericity: chi2={chi2_stat:.1f}, p={p_val:.2e}")

# AFE: entre 2 y min(Kaiser, N_FA-1) — evita sobreparametrizacion
n_factors = int(np.clip(n_factors_kaiser, 2, min(N_FA - 1, 8)))
print(f"\nFactores AFE (Kaiser ajustado): {n_factors}")

# === 3. AFE CON ROTACION VARIMAX (API factor_analyzer) ===
print("\n=== AFE: Cargas rotadas (Varimax) ===")
fa = FactorAnalyzer(n_factors=n_factors, rotation="varimax", method="minres")
try:
    fa.fit(X_fa)
except Exception as e:
    print(f"minres fallo ({e}); reintento con principal")
    fa = FactorAnalyzer(n_factors=n_factors, rotation="varimax", method="principal")
    fa.fit(X_fa)

loadings = fa.loadings_  # (N_FA, n_factors)
# Crear DataFrame de cargas
col_names = [f"F{i+1}" for i in range(n_factors)]
row_names = [f"N{top_idx[j]}" for j in range(N_FA)]
loadings_df = pd.DataFrame(loadings, index=row_names, columns=col_names)

# Marcar cargas significativas (>0.3)
def fmt_load(val):
    s = f"{val:.3f}"
    if abs(val) >= 0.5:
        s = f"**{val:.3f}**"
    elif abs(val) >= 0.3:
        s = f"{val:.3f}*"
    return s

print("\nMatriz de cargas rotadas (con marcas):")
print("  > 0.5: **negrita**, > 0.3: *")
print(loadings_df.to_string())

# Varianza explicada por factor
variance = fa.get_factor_variance()
var_df = pd.DataFrame({
    "SS Loadings": variance[0],
    "Prop Var": variance[1],
    "Cum Var": variance[2],
}, index=col_names)
print("\nVarianza explicada por factor:")
print(var_df.to_string())

# === 4. INDICES AFE (FactorAnalyzer no tiene get_chi2; ver docs) ===
print("\n=== Indices AFE (exploratorio) ===")
n_samples, n_vars = X_fa.shape
R_obs = np.corrcoef(X_fa.T)
communalities = fa.get_communalities()
fa_corr = loadings @ loadings.T + np.diag(communalities)
np.fill_diagonal(fa_corr, 1.0)
res_corr = R_obs - fa_corr
triu = np.triu_indices(n_vars, k=1)
rmsr_efa = float(np.sqrt(np.mean(res_corr[triu] ** 2)))
var_cum = float(variance[2][-1])
print(f"RMSR (matriz residual correlacion): {rmsr_efa:.4f}")
print(f"Varianza acumulada AFE: {var_cum:.1%}")

comm_df = pd.DataFrame({
    "Comunalidad": communalities,
    "Unicidad": 1 - communalities,
}, index=row_names)
print("\nComunalidades (top 10):")
print(comm_df.head(10).to_string())

# === 5. AFC — ConfirmatoryFactorAnalyzer (ML, chi2/RMSEA/CFI) ===
print("\n=== AFC: modelo confirmatorio (4 constructos) ===")
CONSIGNA_FACTORS = [
    "Carga_Antropogenica",
    "Estres_Vegetal",
    "Densidad_Urbana",
    "Volatilidad_Atmosferica",
]

def _cov_to_corr(S):
    d = np.sqrt(np.diag(S))
    d[d < 1e-8] = 1.0
    return S / np.outer(d, d)

def _ml_chi2(S, Sigma, n):
    """Chi-cuadrado ML (Bollen) para matrices de covarianza."""
    p = S.shape[0]
    sign_s, logdet_s = slogdet(S)
    sign_m, logdet_m = slogdet(Sigma)
    if sign_s <= 0 or sign_m <= 0:
        return np.nan, np.nan
    F_ml = logdet_m - logdet_s + np.trace(S @ inv(Sigma)) - p
    chi2 = max(0.0, (n - 1) * F_ml)
    return float(chi2), float(F_ml)

def _cfa_dof(p, n_latent, n_loadings_free):
    n_unique = p * (p + 1) // 2
    n_est = n_loadings_free + p + n_latent + n_latent * (n_latent - 1) // 2
    return max(1, n_unique - n_est)

# AFC: 4 factores fijos; asignar neuronas por cargas de un AFE de 4 factores
fa4 = FactorAnalyzer(n_factors=4, rotation="varimax", method="minres")
fa4.fit(X_fa)
loadings4 = fa4.loadings_
assign = {name: [] for name in CONSIGNA_FACTORS}
for j in range(N_FA):
    f_idx = int(np.argmax(np.abs(loadings4[j])))
    assign[CONSIGNA_FACTORS[f_idx]].append(row_names[j])
for name in CONSIGNA_FACTORS:
    if len(assign[name]) < 2:
        used = {x for vals in assign.values() for x in vals}
        pool = [rn for rn in row_names if rn not in used]
        assign[name].extend(pool[: max(0, 2 - len(assign[name]))])

X_df = pd.DataFrame(X_fa, columns=row_names)
model_spec = ModelSpecificationParser.parse_model_specification_from_dict(X_df, assign)
cfa = ConfirmatoryFactorAnalyzer(model_spec, disp=False)
cfa.fit(X_df.values)

S_sample = np.cov(X_df.values, rowvar=False, bias=False)
Sigma_imp = cfa.get_model_implied_cov()
R_imp = _cov_to_corr(Sigma_imp)
srmr_cfa = float(np.sqrt(np.mean((R_obs - R_imp)[triu] ** 2)))

n_load_free = int(np.count_nonzero(cfa.loadings_))
df_cfa = _cfa_dof(n_vars, len(CONSIGNA_FACTORS), n_load_free)
chi2_cfa, F_ml = _ml_chi2(S_sample, Sigma_imp, n_samples)
rmsea_cfa = float(np.sqrt(max(0.0, (chi2_cfa - df_cfa) / (df_cfa * max(n_samples - 1, 1)))))
chi2_null = _ml_chi2(S_sample, np.diag(np.diag(S_sample)), n_samples)[0]
cfi_cfa = float(np.clip(1.0 - max(chi2_cfa - df_cfa, 0) / max(chi2_null - df_cfa, 1e-6), 0, 1))

print("Asignacion neurona → constructo:")
for k, v in assign.items():
    print(f"  {k}: {len(v)} indicadores")
print(f"SRMR (CFA): {srmr_cfa:.4f}")
print(f"Chi2 (ML): {chi2_cfa:.2f} | gl={df_cfa}")
print(f"RMSEA: {rmsea_cfa:.4f}  (objetivo < 0.08)")
print(f"CFI: {cfi_cfa:.4f}  (objetivo > 0.90)")
print(f"AIC: {cfa.aic_:.2f} | BIC: {cfa.bic_:.2f}")

# === 6. EXPORTAR RESULTADOS ===
results_afe = {
    "config": {
        "n_variables": N_FA,
        "n_factors_afe": n_factors,
        "rotation": "varimax",
        "method_efa": "minres",
        "referencia": "https://factor-analyzer.readthedocs.io/en/latest/",
    },
    "pruebas_idoneidad": {
        "kmo": float(kmo_total),
        "bartlett_chi2": float(chi2_stat),
        "bartlett_p": float(p_val),
    },
    "scree_eigenvalues": ev[:15].tolist(),
    "varianza_explicada": {
        col_names[i]: {
            "ss_loading": float(variance[0][i]),
            "prop_var": float(variance[1][i]),
            "cum_var": float(variance[2][i]),
        }
        for i in range(n_factors)
    },
    "cargas_rotadas": {
        row_names[j]: {col_names[i]: float(loadings[j, i]) for i in range(n_factors)}
        for j in range(N_FA)
    },
    "indices_afe": {
        "rmsr_correlacion_residual": rmsr_efa,
        "varianza_acumulada": var_cum,
    },
    "afc_modelo": assign,
    "indices_afc": {
        "srmr": srmr_cfa,
        "rmsea": rmsea_cfa,
        "cfi": cfi_cfa,
        "chi2_ml": chi2_cfa,
        "df": int(df_cfa),
        "aic": float(cfa.aic_),
        "bic": float(cfa.bic_),
        "log_likelihood": float(cfa.log_likelihood_),
    },
    "top_neuronas": top_idx.tolist(),
}

with open(ANALISIS_DIR / "resultados_afe_afc.json", "w", encoding="utf-8") as f:
    json.dump(results_afe, f, indent=2, ensure_ascii=False)
print(f"\nGuardado: {ANALISIS_DIR / 'resultados_afe_afc.json'}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 kB 1.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Activaciones: (2263, 512)
Top 25 neuronas discriminativas: [452, 36, 19, 163, 179, 406, 47, 413, 13, 142, 262, 349, 286, 76, 94, 281, 330, 306, 304, 473, 188, 332, 426, 105, 50]
Scree plot: /content/runs/sit2_sae/analisis/scree_plot.png

=== Pruebas de idoneidad ===
KMO global: 0.7279
Bartlett sphericity: chi2=22096.9, p=0.00e+00

Factores AFE (Kaiser ajustado): 7

=== AFE: Cargas rotadas (Varimax) ===

Matriz de cargas rotadas (con marcas):
  > 0.5: **negrita**, > 0.3: *
            F1        F2        F3        F4        F5        F6        F7
N452 -0.308304 -0.612581 -0.095081 -0.455963 -0.396049  0.117413 -0.275823
N36   0.247192  0.391784  0.410786  0.563823  0.390437 -0.314513 -0.037049
N19  -0.197918  0.655694 -0.134328  0.249167  0.022654 -0.227815 -0.133355
N163 -0.

/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


Asignacion neurona → constructo:
  Carga_Antropogenica: 13 indicadores
  Estres_Vegetal: 6 indicadores
  Densidad_Urbana: 3 indicadores
  Volatilidad_Atmosferica: 3 indicadores
SRMR (CFA): 0.1490
Chi2 (ML): 12860.09 | gl=265
RMSEA: 0.1450  (objetivo < 0.08)
CFI: 0.4255  (objetivo > 0.90)
AIC: 151542.00 | BIC: 152457.91

Guardado: /content/runs/sit2_sae/analisis/resultados_afe_afc.json


In [8]:
# @title 7. Exportar SAE + embeddings + clip_modelo + zip
import shutil, hashlib, json
from datetime import datetime, timezone
from pathlib import Path

EXP_DIR = Path("/content/sae_modelo_final")
EXP_DIR.mkdir(parents=True, exist_ok=True)

CKPT_SAE = OUT_DIR / "sae_best.pt"
HISTORY = OUT_DIR / "training_history.json"

if not CKPT_SAE.is_file():
    cand = sorted(OUT_DIR.glob("*.pt"), key=os.path.getmtime, reverse=True)
    CKPT_SAE = cand[0] if cand else None
if CKPT_SAE is None:
    raise FileNotFoundError("No se encontro checkpoint SAE en " + str(OUT_DIR))

# SHA256 del checkpoint
h = hashlib.sha256()
with open(CKPT_SAE, "rb") as f:
    for chunk in iter(lambda: f.read(65536), b""):
        h.update(chunk)
ckpt_hash = h.hexdigest()
print(f"Checkpoint: {CKPT_SAE.name} ({CKPT_SAE.stat().st_size / 1e6:.1f} MB)")
print(f"SHA256: {ckpt_hash}")

# Cargar metricas del historial
if HISTORY.is_file():
    with open(HISTORY) as f:
        history = json.load(f)
    best = min(history, key=lambda x: x["mse"]) if history else {}
    final_metrics = {
        "best_mse": best.get("mse"),
        "best_sparsity": best.get("sparsity"),
        "final_mse": history[-1]["mse"] if history else None,
        "final_sparsity": history[-1]["sparsity"] if history else None,
        "epochs_trained": len(history),
    }
else:
    final_metrics = {"error": "training_history.json no encontrado"}

shutil.copy2(CKPT_SAE, EXP_DIR / "sae_best.pt")
if HISTORY.is_file():
    shutil.copy2(HISTORY, EXP_DIR / "training_history.json")

for sub in ("clip_modelo", "embeddings", "analisis"):
    src = OUT_DIR / sub
    if src.is_dir():
        shutil.copytree(src, EXP_DIR / sub, dirs_exist_ok=True)

metadata = {
    "modelo": "SAE (Linear 512 -> ReLU -> Linear 512)",
    "dataset": HF_REPO_ID,
    "checkpoint_clip": str(CKPT),
    "directorios": {
        "clip_modelo": "artefactos de sit2_geovision_clip (best.pt, logs, curvas)",
        "embeddings": "visual_512.pt, sae_latent_512.pt, manifest.json",
        "analisis": "heatmap, AFE, correlaciones, resultados_sae.json",
    },
    "config": {
        "epochs": EPOCHS,
        "lr": LR,
        "lambda_l1": LAMBDA_L1,
        "batch_size": BATCH_SIZE,
        "optimizer": "AdamW",
        "sparsity_target": 0.75,
        "decoder": "sin bias, normalizado por fila post-update",
    },
    "metricas": final_metrics,
    "hash_sae_checkpoint": ckpt_hash,
    "fecha": datetime.now(timezone.utc).isoformat(),
}

with open(EXP_DIR / "metadata_sae.json", "w") as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

# Resumen legible
summary = f"""=== SAE Modelo Final ===
Checkpoint: sae_best.pt ({CKPT_SAE.stat().st_size / 1e6:.1f} MB)
SHA256: {ckpt_hash}

Config:
  Epochs: {metadata['config']['epochs']}
  LR: {metadata['config']['lr']}
  L1 lambda: {metadata['config']['lambda_l1']}
  Batch: {metadata['config']['batch_size']}

Metricas:
  Best MSE:       {final_metrics.get('best_mse', 'N/A')}
  Best Sparsity:  {final_metrics.get('best_sparsity', 'N/A')}
  Final MSE:      {final_metrics.get('final_mse', 'N/A')}
  Final Sparsity: {final_metrics.get('final_sparsity', 'N/A')}

Archivos en el zip:
  - sae_best.pt, training_history.json, metadata_sae.json
  - clip_modelo/ (best.pt y artefactos del notebook CLIP)
  - embeddings/ (visual_512.pt, sae_latent_512.pt, manifest.json)
  - analisis/ (heatmap, scree, AFE, correlaciones)

Verificar integridad:
  certutil -hashfile sae_best.pt SHA256
  Debe dar: {ckpt_hash}
"""
with open(EXP_DIR / "resumen_sae.txt", "w") as f:
    f.write(summary)
print(summary)

shutil.make_archive(str(EXP_DIR), "zip", str(EXP_DIR))
print(f"\nZip listo: {str(EXP_DIR)}.zip")

try:
    from google.colab import drive
    if not Path("/content/drive").is_dir():
        drive.mount("/content/drive", force_remount=False)
    dest = Path("/content/drive/MyDrive/geovision-cali/sit2_sae_posttrain")
    dest.mkdir(parents=True, exist_ok=True)
    shutil.copy2(Path(str(EXP_DIR) + ".zip"), dest / "sae_modelo_final.zip")
    for sub in ("clip_modelo", "embeddings", "analisis"):
        src = EXP_DIR / sub
        if src.is_dir():
            shutil.copytree(src, dest / sub, dirs_exist_ok=True)
    shutil.copy2(EXP_DIR / "sae_best.pt", dest / "sae_best.pt")
    print("Drive:", dest)
except Exception as e:
    print("Drive omitido:", e)

print("Descargar: Files -> /content/sae_modelo_final.zip")

Checkpoint: sae_best.pt (2.1 MB)
SHA256: 8e1989c7f328b2c226c07952c09e1af2e84914009d85f4c7b166c2f458360e58
=== SAE Modelo Final ===
Checkpoint: sae_best.pt (2.1 MB)
SHA256: 8e1989c7f328b2c226c07952c09e1af2e84914009d85f4c7b166c2f458360e58

Config:
  Epochs: 4000
  LR: 0.001
  L1 lambda: 0.005
  Batch: 512

Metricas:
  Best MSE:       1.5115469250304158e-05
  Best Sparsity:  0.7578996181488037
  Final MSE:      1.755239391059149e-05
  Final Sparsity: 0.8134295463562011

Archivos en el zip:
  - sae_best.pt, training_history.json, metadata_sae.json
  - clip_modelo/ (best.pt y artefactos del notebook CLIP)
  - embeddings/ (visual_512.pt, sae_latent_512.pt, manifest.json)
  - analisis/ (heatmap, scree, AFE, correlaciones)

Verificar integridad:
  certutil -hashfile sae_best.pt SHA256
  Debe dar: 8e1989c7f328b2c226c07952c09e1af2e84914009d85f4c7b166c2f458360e58


Zip listo: /content/sae_modelo_final.zip
Drive: /content/drive/MyDrive/geovision-cali/sit2_sae_posttrain
Descargar: Files -> /content